# Banglore Price prediction

**INTRODUCTION**:
Property valuation is the backbone of every strategic decision in the real estate sector. A fundamental metric of a real estate firm's performance or an investor's success is the capacity to accurately estimate property values and maximize investment returns. Looking at raw market listings alone, however, is insufficient. It's crucial to make a thorough study of historical housing data and property attributes to determine how different features drive market value. Market trends, neighborhood demand, and pricing anomalies can all be better understood through this analysis and subsequent predictive modeling. We'll see the value of machine learning in real estate in this project, along with how it may assist developers, buyers, and agents in making wise financial decisions and mitigating investment risks.

**PROJECT STATEMENT**:
I've been given access to Bangalore real estate transaction data, and I'm required to draw insightful conclusions from the information and build a predictive machine learning model to estimate housing prices. The information is shown in a structured dataset that includes columns for property features such as area_type, availability, location, size, society, total_sqft, bath, balcony, and price.

**AIM**:
I am to locate the specific neighborhoods or zones that yield the highest property appreciation, identify the core structural and locational attributes most desired by homebuyers, and pinpoint peak buying seasons across the months of the year. Furthermore, I aim to analyze the direct relationship between physical property dimensions and final valuations, handle missing data or outliers to prevent skewed predictions, and leverage algorithms like scikit-learn's linear or tree-based regressions to automate accurate pricing. 

**Importing important libraries needed for the Project**

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import ridge_regression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix 
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import ShuffleSplit
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor


**Import the data**: Here i imported the data and performed pre-processing analysis, to get the data structure.

In [27]:
df = pd.read_csv("C:\\Users\\DELL\\Downloads\\banglore.csv")
df

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00
...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453,4.0,0.0,231.00
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600,5.0,NaN,400.00
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141,2.0,1.0,60.00
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689,4.0,1.0,488.00


In [28]:
df.shape

(13320, 9)

In [29]:
df.iloc[5771]

area_type       Super built-up  Area
availability           Ready To Move
location         Devarachikkanahalli
size                           3 BHK
society                      Shireru
total_sqft                      1425
bath                             2.0
balcony                          2.0
price                           65.0
Name: 5771, dtype: object

In [30]:
df.groupby(by ="area_type")["area_type"].count()

area_type
Built-up  Area          2418
Carpet  Area              87
Plot  Area              2025
Super built-up  Area    8790
Name: area_type, dtype: int64

In [31]:
df.drop(["availability", "society", "balcony", "area_type"], axis =1, inplace= True)

In [32]:
df.isnull().sum()

location       1
size          16
total_sqft     0
bath          73
price          0
dtype: int64

**Data Cleaning & Preprocessing**

To ensure data quality and formatting consistency before model training, the following preprocessing steps were performed:

• Handling Missing Values: Dropped all observations containing null values to maintain a complete and robust dataset.

• Feature Standardisation (size): Converted the size column into a standardized float format to represent the bedroom/room count numerically.

• Standardising Spatial Data (total_sqft): Resolved formatting inconsistencies where square footage was displayed as a range (e.g., "2100 - 2850"). These ranges were parsed and converted into a single float value by calculating the average of the range bounds.


In [33]:
df.dropna(inplace = True)

In [34]:
df

,location,size,total_sqft,bath,price
0,Electronic City Phase II,2 BHK,1056,2.0,39.07
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00
2,Uttarahalli,3 BHK,1440,2.0,62.00
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00
4,Kothanur,2 BHK,1200,2.0,51.00
...,...,...,...,...,...
13315,Whitefield,5 Bedroom,3453,4.0,231.00
13316,Richards Town,4 BHK,3600,5.0,400.00
13317,Raja Rajeshwari Nagar,2 BHK,1141,2.0,60.00
13318,Padmanabhanagar,4 BHK,4689,4.0,488.00


In [35]:
df["size"].unique()

<StringArray>
[     '2 BHK',  '4 Bedroom',      '3 BHK',      '4 BHK',  '6 Bedroom',
  '3 Bedroom',      '1 BHK',       '1 RK',  '1 Bedroom',  '8 Bedroom',
  '2 Bedroom',  '7 Bedroom',      '5 BHK',      '7 BHK',      '6 BHK',
  '5 Bedroom',     '11 BHK',      '9 BHK',  '9 Bedroom',     '27 BHK',
 '10 Bedroom', '11 Bedroom',     '10 BHK',     '19 BHK',     '16 BHK',
 '43 Bedroom',     '14 BHK',      '8 BHK', '12 Bedroom',     '13 BHK',
 '18 Bedroom']
Length: 31, dtype: str

In [36]:
df.info()

<class 'pandas.DataFrame'>
Index: 13246 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13246 non-null  str    
 1   size        13246 non-null  str    
 2   total_sqft  13246 non-null  str    
 3   bath        13246 non-null  float64
 4   price       13246 non-null  float64
dtypes: float64(2), str(3)
memory usage: 620.9 KB


In [37]:
def alt_size(x):
    res = x.split(' ')
    return float(res[0])
alt_size("2")

2.0

In [38]:
df["size"] = df["size"].apply(alt_size)

In [39]:
df

,location,size,total_sqft,bath,price
0,Electronic City Phase II,2.0,1056,2.0,39.07
1,Chikka Tirupathi,4.0,2600,5.0,120.00
2,Uttarahalli,3.0,1440,2.0,62.00
3,Lingadheeranahalli,3.0,1521,3.0,95.00
4,Kothanur,2.0,1200,2.0,51.00
...,...,...,...,...,...
13315,Whitefield,5.0,3453,4.0,231.00
13316,Richards Town,4.0,3600,5.0,400.00
13317,Raja Rajeshwari Nagar,2.0,1141,2.0,60.00
13318,Padmanabhanagar,4.0,4689,4.0,488.00


In [40]:
numeric_column = pd.to_numeric(df['total_sqft'], errors='coerce')

# Check for non-float values
non_float_values = df[pd.isna(numeric_column)]

print("Non-float values in the column:")
print(non_float_values)

Non-float values in the column:
                 location  size   total_sqft  bath    price
30              Yelahanka   4.0  2100 - 2850   4.0  186.000
122                Hebbal   4.0  3067 - 8156   4.0  477.000
137    8th Phase JP Nagar   2.0  1042 - 1105   2.0   54.005
165              Sarjapur   2.0  1145 - 1340   2.0   43.490
188              KR Puram   2.0  1015 - 1540   2.0   56.800
...                   ...   ...          ...   ...      ...
12975          Whitefield   2.0   850 - 1060   2.0   38.190
12990      Talaghattapura   3.0  1804 - 2273   3.0  122.000
13059              Harlur   2.0  1200 - 1470   2.0   72.760
13265               Hoodi   2.0  1133 - 1384   2.0   59.135
13299          Whitefield   4.0  2830 - 2882   5.0  154.500

[190 rows x 5 columns]


In [41]:
def alt_sqft(x):
    tokens = x.split('-')
    if len(tokens) == 2:
        return (float(tokens[0])+float(tokens[1]))/2
    try:
        return float(x)
    except:
        return None   

In [42]:
df["total_sqft"] = df["total_sqft"].apply(alt_sqft)
df

,location,size,total_sqft,bath,price
0,Electronic City Phase II,2.0,1056.0,2.0,39.07
1,Chikka Tirupathi,4.0,2600.0,5.0,120.00
2,Uttarahalli,3.0,1440.0,2.0,62.00
3,Lingadheeranahalli,3.0,1521.0,3.0,95.00
4,Kothanur,2.0,1200.0,2.0,51.00
...,...,...,...,...,...
13315,Whitefield,5.0,3453.0,4.0,231.00
13316,Richards Town,4.0,3600.0,5.0,400.00
13317,Raja Rajeshwari Nagar,2.0,1141.0,2.0,60.00
13318,Padmanabhanagar,4.0,4689.0,4.0,488.00


In [43]:
df[df.duplicated()]

,location,size,total_sqft,bath,price
242,Chandapura,1.0,645.0,1.0,16.45
887,Chandapura,2.0,1015.0,2.0,25.88
895,Whitefield,2.0,1116.0,2.0,51.91
905,Kanakpura Road,1.0,525.0,1.0,26.00
971,Haralur Road,3.0,1464.0,3.0,56.00
...,...,...,...,...,...
13291,Weavers Colony,1.0,812.0,1.0,26.00
13299,Whitefield,4.0,2856.0,5.0,154.50
13311,Ramamurthy Nagar,7.0,1500.0,9.0,250.00
13313,Uttarahalli,3.0,1345.0,2.0,57.00


In [44]:
df["total_sqft"].unique()

array([1056. , 2600. , 1440. , ..., 1258.5,  774. , 4689. ], shape=(1973,))

In [45]:
df["location"].unique().shape[0]

1304

**Exploratory Data Analysis (EDA) & Feature Engineering**

• Profiled property distribution across neighborhoods, identifying Whitefield as the location with the highest house count.

• Sanitized the dataset by removing statistical outliers in total_sqft and size where maximum values heavily skewed the distribution mean.

• Converted textual categorical data (location) into numerical values using pandas get_dummies for model compatibility.

In [46]:
df.groupby(by = "location")["location"].count().sort_values(ascending = False)

location
Whitefield                                            534
Sarjapur  Road                                        392
Electronic City                                       302
Kanakpura Road                                        266
Thanisandra                                           233
                                                     ... 
bsk 6th stage 2ad block near sri conversation hall      1
beml layout, basaveshwara nagar                         1
basaveshwarnagar                                        1
banashankari stage iii sa                               1
asha township, off hennur road                          1
Name: location, Length: 1304, dtype: int64

In [47]:
item_counts = df['location'].value_counts()
items_to_replace = item_counts[item_counts <= 10].index.tolist()
df['location'] = df['location'].replace(items_to_replace, "other")
df

,location,size,total_sqft,bath,price
0,Electronic City Phase II,2.0,1056.0,2.0,39.07
1,Chikka Tirupathi,4.0,2600.0,5.0,120.00
2,Uttarahalli,3.0,1440.0,2.0,62.00
3,Lingadheeranahalli,3.0,1521.0,3.0,95.00
4,Kothanur,2.0,1200.0,2.0,51.00
...,...,...,...,...,...
13315,Whitefield,5.0,3453.0,4.0,231.00
13316,other,4.0,3600.0,5.0,400.00
13317,Raja Rajeshwari Nagar,2.0,1141.0,2.0,60.00
13318,Padmanabhanagar,4.0,4689.0,4.0,488.00


In [48]:
item_counts

location
Whitefield                                         534
Sarjapur  Road                                     392
Electronic City                                    302
Kanakpura Road                                     266
Thanisandra                                        233
                                                  ... 
Pattegarhpalya                                       1
Tilak Nagar                                          1
12th cross srinivas nagar banshankari 3rd stage      1
Havanur extension                                    1
Abshot Layout                                        1
Name: count, Length: 1304, dtype: int64

In [49]:
df["location"].unique().shape[0]

242

In [50]:
df["price_per_sqft"] = (df["price"]*100000)/df["total_sqft"]

In [51]:
df.describe()

,size,total_sqft,bath,price,price_per_sqft
count,13246.000000,13200.000000,13246.000000,13246.000000,1.320000e+04
mean,2.801902,1555.302783,2.692586,112.389392,7.920759e+03
std,1.295758,1237.323445,1.341506,149.076587,1.067272e+05
min,1.000000,1.000000,1.000000,8.000000,2.678298e+02
25%,2.000000,1100.000000,2.000000,50.000000,4.267701e+03
50%,3.000000,1275.000000,2.000000,72.000000,5.438331e+03
75%,3.000000,1672.000000,3.000000,120.000000,7.317073e+03
max,43.000000,52272.000000,40.000000,3600.000000,1.200000e+07


In [52]:
df.shape

(13246, 6)

In [53]:
df1 = df[~(df["total_sqft"]/df["size"]<300)]
df1.shape

(12502, 6)

In [54]:
df1.describe()

,size,total_sqft,bath,price,price_per_sqft
count,12502.000000,12456.000000,12502.000000,12502.000000,12456.000000
mean,2.650696,1590.189927,2.564790,111.311915,6308.502826
std,0.981698,1260.404795,1.084946,152.089966,4168.127339
min,1.000000,300.000000,1.000000,9.000000,267.829813
25%,2.000000,1115.000000,2.000000,49.000000,4210.526316
50%,3.000000,1300.000000,2.000000,70.000000,5294.117647
75%,3.000000,1700.000000,3.000000,115.000000,6916.666667
max,16.000000,52272.000000,16.000000,3600.000000,176470.588235


In [55]:
def remove_pps_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        reduced_df = subdf[(subdf.price_per_sqft>(m-st)) & (subdf.price_per_sqft<=(m+st))]
        df_out = pd.concat([df_out,reduced_df],ignore_index=True)
    return df_out



In [56]:
df2 = remove_pps_outliers(df1)
df2.shape

(10244, 6)

In [57]:
df2.describe()

,size,total_sqft,bath,price,price_per_sqft
count,10244.000000,10244.000000,10244.000000,10244.000000,10244.000000
mean,2.571847,1503.662115,2.473741,90.968580,5657.802131
std,0.896375,876.694693,0.981421,86.139083,2266.273845
min,1.000000,300.000000,1.000000,10.000000,1250.000000
25%,2.000000,1108.000000,2.000000,49.000000,4244.178277
50%,2.000000,1282.000000,2.000000,67.000000,5172.846776
75%,3.000000,1650.000000,3.000000,100.000000,6426.419630
max,16.000000,30400.000000,16.000000,2200.000000,24509.803922


In [58]:
def remove_bhk_outliers(df):
    exclude_indices = np.array([])
    for location, location_df in df.groupby('location'):
        size_stats = {}
        for size, size_df in location_df.groupby('size'):
            size_stats[size] = {
                'mean': np.mean(size_df.price_per_sqft),
                'std': np.std(size_df.price_per_sqft),
                'count': size_df.shape[0]
            }
        for size, size_df in location_df.groupby('size'):
            stats = size_stats.get(size-1)
            if stats and stats['count']>5:
                exclude_indices = np.append(exclude_indices, size_df[size_df.price_per_sqft<(stats['mean'])].index.values)
    return df.drop(exclude_indices,axis='index')

df4 = remove_bhk_outliers(df2)

df4

,location,size,total_sqft,bath,price,price_per_sqft
1,Devarachikkanahalli,2.0,1250.0,2.0,40.0,3200.000000
2,Devarachikkanahalli,2.0,1200.0,2.0,83.0,6916.666667
3,Devarachikkanahalli,2.0,1170.0,2.0,40.0,3418.803419
4,Devarachikkanahalli,3.0,1425.0,2.0,65.0,4561.403509
5,Devarachikkanahalli,2.0,947.0,2.0,43.0,4540.654699
...,...,...,...,...,...,...
10235,other,2.0,1200.0,2.0,70.0,5833.333333
10236,other,1.0,1800.0,1.0,200.0,11111.111111
10239,other,2.0,1353.0,2.0,110.0,8130.081301
10240,other,1.0,812.0,1.0,26.0,3201.970443


In [59]:
df5 = df4[~((df4["bath"])> (df4["size"]+2))]
df5

,location,size,total_sqft,bath,price,price_per_sqft
1,Devarachikkanahalli,2.0,1250.0,2.0,40.0,3200.000000
2,Devarachikkanahalli,2.0,1200.0,2.0,83.0,6916.666667
3,Devarachikkanahalli,2.0,1170.0,2.0,40.0,3418.803419
4,Devarachikkanahalli,3.0,1425.0,2.0,65.0,4561.403509
5,Devarachikkanahalli,2.0,947.0,2.0,43.0,4540.654699
...,...,...,...,...,...,...
10235,other,2.0,1200.0,2.0,70.0,5833.333333
10236,other,1.0,1800.0,1.0,200.0,11111.111111
10239,other,2.0,1353.0,2.0,110.0,8130.081301
10240,other,1.0,812.0,1.0,26.0,3201.970443


In [60]:
df6 = df5.drop("price_per_sqft", axis = 1)
df6

,location,size,total_sqft,bath,price
1,Devarachikkanahalli,2.0,1250.0,2.0,40.0
2,Devarachikkanahalli,2.0,1200.0,2.0,83.0
3,Devarachikkanahalli,2.0,1170.0,2.0,40.0
4,Devarachikkanahalli,3.0,1425.0,2.0,65.0
5,Devarachikkanahalli,2.0,947.0,2.0,43.0
...,...,...,...,...,...
10235,other,2.0,1200.0,2.0,70.0
10236,other,1.0,1800.0,1.0,200.0
10239,other,2.0,1353.0,2.0,110.0
10240,other,1.0,812.0,1.0,26.0


In [61]:
df7 = pd.get_dummies(df6["location"])
df7

,Devarachikkanahalli,1st Block Jayanagar,1st Phase JP Nagar,2nd Phase Judicial Layout,2nd Stage Nagarbhavi,5th Block Hbr Layout,5th Phase JP Nagar,6th Phase JP Nagar,7th Phase JP Nagar,8th Phase JP Nagar,...,Vishveshwarya Layout,Vishwapriya Layout,Vittasandra,Whitefield,Yelachenahalli,Yelahanka,Yelahanka New Town,Yelenahalli,Yeshwanthpur,other
1,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10235,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
10236,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
10239,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
10240,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [62]:
df8 = pd.concat([df6.drop(["location"], axis=1), df7.drop(["other"], axis =1)], axis =1)
df8

,size,total_sqft,bath,price,Devarachikkanahalli,1st Block Jayanagar,1st Phase JP Nagar,2nd Phase Judicial Layout,2nd Stage Nagarbhavi,5th Block Hbr Layout,...,Vijayanagar,Vishveshwarya Layout,Vishwapriya Layout,Vittasandra,Whitefield,Yelachenahalli,Yelahanka,Yelahanka New Town,Yelenahalli,Yeshwanthpur
1,2.0,1250.0,2.0,40.0,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2.0,1200.0,2.0,83.0,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2.0,1170.0,2.0,40.0,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,3.0,1425.0,2.0,65.0,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5,2.0,947.0,2.0,43.0,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10235,2.0,1200.0,2.0,70.0,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
10236,1.0,1800.0,1.0,200.0,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
10239,2.0,1353.0,2.0,110.0,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
10240,1.0,812.0,1.0,26.0,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


**Model Building & Deployment**
• Partitioned data into training (80%) and testing (20%) subsets using train_test_split.
• Evaluated three different algorithms: Linear Regression, Lasso, and Decision Tree Regressor.
• Selected Linear Regression as the optimal model after it yielded the best accuracy score.
• Serialized the trained model into a pickle file for smooth integration into the web application's backend.

In [63]:
X = df8.drop("price", axis =1)
y= df8["price"]

In [64]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size =0.2, random_state =20)

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](244,)","[ -3.6 , 0.08, 4.26,...,-25.56,-51.51,-17.33]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](244,)","['size','total_sqft','bath',...,'Yelahanka New Town','Yelenahalli', 'Yeshwanthpur']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-2.481
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,244
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(244)


In [65]:
model.score(X_test, y_test)

0.8443979303373879

In [66]:
X_test

,size,total_sqft,bath,Devarachikkanahalli,1st Block Jayanagar,1st Phase JP Nagar,2nd Phase Judicial Layout,2nd Stage Nagarbhavi,5th Block Hbr Layout,5th Phase JP Nagar,...,Vijayanagar,Vishveshwarya Layout,Vishwapriya Layout,Vittasandra,Whitefield,Yelachenahalli,Yelahanka,Yelahanka New Town,Yelenahalli,Yeshwanthpur
2588,2.0,1140.0,2.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4528,3.0,1423.0,2.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3470,3.0,1200.0,3.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3491,3.0,1920.0,3.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4511,2.0,1140.0,2.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3695,2.0,918.0,2.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2384,2.0,1150.0,2.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
7231,2.0,1485.0,2.0,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
2549,2.0,1467.0,2.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [67]:
y_test

2588     46.00
4528     85.00
3470     56.12
3491    130.00
4511     50.66
         ...  
3695     49.00
2384     52.75
7231     90.00
2549     55.00
5681     98.00
Name: price, Length: 1468, dtype: float64

In [68]:
cv = ShuffleSplit(n_splits=5,  random_state=0)
print(cross_val_score(model, X, y, cv=cv))

[0.80132195 0.86208173 0.82900574 0.78009521 0.76498739]


In [69]:
all_model_para = {
    "reg":{
        "model": LinearRegression(),
        "param": {'fit_intercept': [True, False]}
            }, 
    "lasso":{
        "model": Lasso(),
        "param": {'alpha': [1,2],
                'selection': ['random', 'cyclic']}, 
    }, 
    "deci": {
        "model":DecisionTreeRegressor(),
        "param":{'criterion' : ['mse','friedman_mse'],
                'splitter': ['best','random']}
    },
    
        
    
    
}

In [70]:
score = {"model":[], "best_score":[], "best_param":[]}
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)
for model_name, model_content in all_model_para.items():
    grid = GridSearchCV(model_content["model"], model_content["param"], cv =cv)
    grid.fit(X,y)
    score["model"].append(model_name)
    score["best_score"].append(grid.best_score_)
    score["best_param"].append(grid.best_params_)
    

score

ValueError: 
All the 20 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py", line 1393, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py", line 554, in _validate_params
    validate_parameter_constraints(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        self

In [71]:
score_df = pd.DataFrame(score)
score_df

,model,best_score,best_param
0,reg,0.822279,{'fit_intercept': False}
1,lasso,0.691562,"{'alpha': 1, 'selection': 'random'}"


In [72]:
#so we are adopting the regresion model

In [73]:
def predict_price(size,total_sqft,bath,location):    
    loc_index = np.where(X.columns==location)[0][0]

    x = np.zeros(len(X.columns))
    x[0] = size
    x[1] = total_sqft
    x[2] = bath
    if loc_index >= 0:
        x[loc_index] = 1
            
    return model.predict([x])[0]

 

In [74]:
y_pred = predict_price(3,1425,2," Devarachikkanahalli")
y_pred

C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


np.float64(69.1858743248684)

In [75]:
y_pred = predict_price(3,3400,2,"Vishveshwarya Layout")
y_pred

C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


np.float64(212.0361924487267)

In [76]:
X.columns

Index(['size', 'total_sqft', 'bath', ' Devarachikkanahalli',
       '1st Block Jayanagar', '1st Phase JP Nagar',
       '2nd Phase Judicial Layout', '2nd Stage Nagarbhavi',
       '5th Block Hbr Layout', '5th Phase JP Nagar',
       ...
       'Vijayanagar', 'Vishveshwarya Layout', 'Vishwapriya Layout',
       'Vittasandra', 'Whitefield', 'Yelachenahalli', 'Yelahanka',
       'Yelahanka New Town', 'Yelenahalli', 'Yeshwanthpur'],
      dtype='str', length=244)

In [140]:
import pickle

# Assuming 'model' is your trained model
# Save the model to a file
with open('price_prediction.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model saved as 'model.pkl'")

Model saved as 'model.pkl'


In [147]:
import json

lowercase_columns = [col.lower() for col in X.columns]

# Create a dictionary with the key "data_column" and the lowercase column names as the value
data_dict = {"data_column": lowercase_columns}

# Save the dictionary to a JSON file
with open('columns.json', 'w') as f:
    json.dump(data_dict, f)

print("Columns saved as 'columns.json'")

Columns saved as 'columns.json'
